## Processing Word Documents (.docx)

Word documents present a different set of parsing challenges compared to PDFs, since `.docx` files are actually structured XML packages (not fixed-layout like PDFs). This makes some things easier, but introduces its own quirks.

### Why Word documents are different from PDFs
- **Structured content**: Text is tagged with styles (Heading 1, Normal, List Bullet, etc.), so headings and structure are often recoverable — unlike PDFs where layout must be inferred.
- **No fixed page geometry**: There's no absolute x/y coordinate system, so spatial reasoning (columns, positioning) doesn't apply the same way.
- **Rich embedded objects**: Tables, images, headers/footers, footnotes, comments, and tracked changes all live in separate XML parts within the `.docx` zip archive.

### Common challenges
1. **Tables** — nested tables and merged cells can break naive text extraction; tables often need to be parsed separately from body text.
2. **Headers/footers & footnotes** — may or may not be relevant to your RAG index; easy to accidentally include boilerplate (page numbers, confidentiality notices) as noise.
3. **Embedded images** — text inside images (scanned content, screenshots) won't be extracted without OCR.
4. **Tracked changes & comments** — if not stripped, deleted text or reviewer comments can pollute the extracted content.
5. **Styles vs. semantics** — a heading styled visually (bold + large font) but not tagged as "Heading" won't be detected as structure automatically.
6. **Lists** — bullet/numbered lists may extract as flat paragraphs, losing hierarchy.

### Common tools/libraries
- **`python-docx`** — low-level, gives full control over paragraphs, runs, styles, and tables; good when you need custom logic.
- **`docx2txt`** — quick plain-text + image extraction, minimal structure preservation.
- **`unstructured`** (`partition_docx`) — auto-detects titles, list items, tables, and narrative text; integrates well with chunking pipelines.
- **LangChain loaders** — `Docx2txtLoader` (simple) or `UnstructuredWordDocumentLoader` (structure-aware).
- **LlamaIndex** — `DocxReader` for similar structure-aware parsing.

### Best practices for RAG ingestion
- Preserve heading hierarchy where possible — it makes excellent metadata for chunking and retrieval filtering.
- Extract tables separately and consider serializing them (e.g., to markdown or key-value text) rather than flattening to plain text.
- Strip or explicitly decide how to handle headers/footers, footnotes, and comments.
- Normalize whitespace — Word documents often have inconsistent spacing from manual formatting.


In [1]:
## PDf parsing

from pathlib import Path
import os

# Move to the project root, regardless of current notebook location
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Using DocxtxtLoader
from langchain_community.document_loaders import Docx2txtLoader, UnstructuredWordDocumentLoader

print("Using Docx2textLoader")

try:
    docx_loader = Docx2txtLoader("data/raw/word_files/proposal.docx")
    docs= docx_loader.load()
    print(f" Loaded {len(docs)} document(s)")
    print(f"Content preview : {docs[0].page_content[:200]}...")
    print(f"Metadata: {docs[0].metadata}")
except Exception as e:
    print(f"Error: {e}")

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_25696/1338612136.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader, UnstructuredWordDocumentLoader
/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Docx2textLoader
 Loaded 1 document(s)
Content preview : Project Proposal: RAG Implementation

Executive Summary

This proposal outlines the implementation of a Retrieval-Augmented Generation system for our organization.

Objectives

Key objectives include:...
Metadata: {'source': 'data/raw/word_files/proposal.docx'}


In [3]:
## Using UnstructuredWordDocumentLoader

print(f"Using  Unstructured Word Document Loader Loader")

try:
    unstructured_loader = UnstructuredWordDocumentLoader("data/raw/word_files/proposal.docx", mode="elements")
    unstructured_docs= unstructured_loader.load()

    print(f" Loaded {len(unstructured_docs)} elements")
    
    for i, doc in enumerate(unstructured_docs[:3]):
        print(f"\nElements {i+1}:")
        print(f"Type: {doc.metadata.get('category','unknown')}")
        print(f"Content preview : {doc.page_content[:200]}...")
    
except Exception as e:
    print(f"Error: {e}")

Using  Unstructured Word Document Loader Loader
 Loaded 20 elements

Elements 1:
Type: Title
Content preview : Project Proposal: RAG Implementation...

Elements 2:
Type: Title
Content preview : Executive Summary...

Elements 3:
Type: NarrativeText
Content preview : This proposal outlines the implementation of a Retrieval-Augmented Generation system for our organization....


In [4]:
unstructured_docs

[Document(metadata={'source': 'data/raw/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/raw/word_files', 'filename': 'proposal.docx', 'last_modified': '2026-09-22T20:46:09', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'Title', 'element_id': 'bb0410bfd160ef866f8d4357b0949db2'}, page_content='Project Proposal: RAG Implementation'),
 Document(metadata={'source': 'data/raw/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/raw/word_files', 'filename': 'proposal.docx', 'last_modified': '2026-09-22T20:46:09', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'Title', 'element_id': 'c0f844859abf08d9506856b3aed4a719'}, page_content='Executive Summary'),
 Document(metadata={'source': 'data/raw/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/raw/word_files', 'filename': 'proposal.do